# Sepsis Early-Warning & Escalation Copilot — measured walkthrough

This notebook reads the **committed artefacts** and shows the measured model,
operating point, fairness/drift, grounding, agent boundary, and service results.
It makes no network calls and needs no key.

**Honest scope:** a retrospective benchmark and a portfolio-grade local service,
not a clinical device. See the README for the public summary; the detailed report is kept private.


In [1]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def load(name):
    return json.loads((ROOT / "artifacts" / name).read_text(encoding="utf-8"))

print("artifact root:", ROOT / "artifacts")


artifact root: C:\Users\pravi\Documents\Default Project\sepsis-ews\artifacts


## 1. Data profile

In [2]:
profile = load("data_profile.json")
pd.DataFrame([
    {"patients": profile["totals"]["patients"],
     "patient_hours": profile["totals"]["hours"],
     "sepsis_patients": profile["totals"]["sepsis_patients"],
     "prevalence": profile["totals"]["prevalence"]},
])
pd.DataFrame(profile["sets"]).T


,hours,patients,sepsis_patients
A,790215,20336,1790
B,761995,20000,1142


## 2. Model and operating point

In [3]:
metrics = load("metrics.json")
comparison = load("comparison.json")

pd.DataFrame([
    {"model": "LightGBM", **comparison["lightgbm"]},
    {"model": "GRU 24h", **comparison["gru"]},
])
print("champion:", comparison["champion"])
pd.Series({
    "auroc (pre-onset)": metrics["auroc"],
    "auprc": metrics["auprc"],
    "prevalence": metrics["prevalence"],
    "patient-level auroc": metrics["patient_level_auroc"],
    "brier": metrics["calibration"]["brier"],
    "calibration slope": metrics["calibration"]["slope"],
}, name="test")
pd.Series(metrics["policy"], name="validation-selected policy")


champion: lightgbm


alerts_per_patient_day        1.4630
median_lead_hours             7.0000
n_alerts                  14291.0000
n_early_detected            351.0000
n_eligible                  369.0000
n_late                        0.0000
n_missed                     12.0000
n_on_time                     6.0000
n_patients                 6050.0000
patient_days               9767.9600
sensitivity                   0.9512
threshold                     0.0062
Name: validation-selected policy, dtype: float64

## 3. Fairness and drift

In [4]:
fair = load("fairness.json")
pd.DataFrame(fair["deltas"]).T
drift = load("drift_summary.json")
pd.DataFrame(drift["features"])[["feature", "psi", "verdict"]]
print("recommend_retrain:", drift["recommend_retrain"])


recommend_retrain: False


## 4. Grounding and retrieval

In [5]:
scores = load("eval_scores.json")
pd.Series({
    "faithfulness": scores["faithfulness"],
    "context precision": scores["context_precision"],
    "context recall": scores["context_recall"],
    "answer relevancy": scores["answer_relevancy"],
    "citation validity": scores["citation_validity"],
    "refusal correctness": scores["refusal_correctness"],
    "retrieval recall@5": scores["retrieval_recall_at_5"],
    "false refusal rate": scores["false_refusal_rate"],
}, name="RAGAS / retrieval")
retrieval = load("retrieval_scores.json")
print("retrieval queries:", retrieval["n_queries"],
      "recall@k:", retrieval["recall_at_k"], "MRR:", retrieval["mrr"])


retrieval queries: 20 recall@k: 1.0 MRR: 0.916667


## 5. Agent boundary and the demo case

In [6]:
demo = load("demo_run.json")
state = demo["state"]
print("agent status:", state["status"])
print("generator:", state["generator"])
print("validation:", state["validation"])
for section in ("situation", "background", "assessment", "recommendation"):
    print(f"\n{section.upper()}: {state['sbar'][section]}")
print("\npage:", demo.get("page", {}).get("status"),
      "via", demo.get("page", {}).get("transport"))
print("denied without token:", bool(demo.get("denied_without_token")))


agent status: pending_approval
generator: llm
validation: {'status': 'valid', 'flags': []}

SITUATION: Patient p000206 is currently in the ICU at hour 15 with a calibrated sepsis risk of 0.0391, which is above the operating threshold of 0.0062. Top model drivers include elevated heart rate and creatinine levels.

BACKGROUND: The patient's vital signs include a heart rate of 109.0, O2 saturation of 92.0, temperature of 36.22, mean arterial pressure of 84.0, respiratory rate of 21.0, creatinine of 5.1, WBC of 7.4, platelets of 75.0, and BUN of 108.0.

ASSESSMENT: The calibrated sepsis risk and the patient's elevated heart rate and creatinine levels are concerning for evolving sepsis [S4].

RECOMMENDATION: Consider initiating the Hour-1 bundle for sepsis management, which includes measuring lactate levels, obtaining blood cultures, administering broad-spectrum antibiotics, and fluid resuscitation if hypotensive [S1].

page: paged via local-demo
denied without token: True


## 6. Service and integration evidence

In [7]:
latency = load("latency.json")
print("model p95 ms:", latency["model_score"]["p95_ms"])
print("in-process API p95 ms:", latency["api_score"]["p95_ms"])
stream = load("stream_check.json")
print("48h SSE:", stream["events"], "events in", stream["duration_seconds"],
      "s;", stream["ms_per_event"], "ms/event")
mcp = load("mcp_tools.json")
print("MCP tools:", mcp["tool_count"], "forbidden:", mcp["forbidden_present"])
print("MCP write without token blocked:", mcp["write_without_token_blocked"])
for name in ("oidc_check.json", "p1_e1_enable.json", "fhir_check.json"):
    path = ROOT / "artifacts" / name
    if path.exists():
        print(name, "present")


model p95 ms: 10.039
in-process API p95 ms: 20.534
48h SSE: 48 events in 0.598 s; 12.45 ms/event
MCP tools: 9 forbidden: []
MCP write without token blocked: True
oidc_check.json present
p1_e1_enable.json present
fhir_check.json present


## 7. Reproduce and limits

```bash
python scripts/download_data.py
python -m src.build_gold --raw data/raw
python -m src.train
python -m src.audit
python -m src.rag --build
python -m src.eval_rag
python -m pytest -q
python -m src.gates
```

**Limits:** retrospective PhysioNet 2019 cohort; no external/prospective
validation; AUPRC is low in absolute terms; RAG answer relevancy is 0.607 with
two conservative refusals; no public host or production EHR/pager. T1/P1/E1 are
demo-tier (Dex login, webhook, public HAPI sandbox).
